In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single-1.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed123_d_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/Orginal.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed2026_a.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single-2.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed42_c_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed3407_b_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed2026_a_raw.csv
/

## Voting 

In [2]:
import pandas as pd
import numpy as np
 
COMP = '/kaggle/input/competitions/playground-series-s6e4/'
DS   = '/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/'
 
sub = pd.read_csv(COMP + 'sample_submission.csv')
 
# 4 raw seeds for agreement detection
a = pd.read_csv(DS + 'submission_seed2026_a_raw.csv').rename(columns={'Irrigation_Need':'A'})
b = pd.read_csv(DS + 'submission_seed3407_b_raw.csv').rename(columns={'Irrigation_Need':'B'})
c = pd.read_csv(DS + 'submission_seed42_c_raw.csv').rename(columns={'Irrigation_Need':'C'})
d = pd.read_csv(DS + 'submission_seed123_d_raw.csv').rename(columns={'Irrigation_Need':'D'})
 
# Fallback = our BEST submission 
best = pd.read_csv(DS + 'best_single-2.csv').rename(columns={'Irrigation_Need':'BEST'})
 
dfs = a.merge(b, on='id').merge(c, on='id').merge(d, on='id').merge(best, on='id')
 
# Agreement check
dfs['all4_agree'] = (dfs['A']==dfs['B']) & (dfs['B']==dfs['C']) & (dfs['C']==dfs['D'])
print(f"All 4 agree: {dfs['all4_agree'].sum():,} / {len(dfs):,}")
print(f"Disagree:    {(~dfs['all4_agree']).sum():,}")
 
# Vote: consensus when agree, best Optuna model when disagree
def vote(row):
    if row['all4_agree']:
        return row['A']
    return row['BEST']   # Optuna-weighted seed2026 on uncertain rows
 
sub['Irrigation_Need'] = dfs.apply(vote, axis=1)
sub.to_csv('submission_voting_v2.csv', index=False)
print(f"\nDist: {sub['Irrigation_Need'].value_counts().to_dict()}")
print("Saved submission_voting_v2.csv")
 
# ── Also try: majority vote among the 4 seeds (no fallback) ──────────
def majority_vote(row):
    votes = [row['A'], row['B'], row['C'], row['D']]
    return max(set(votes), key=votes.count)
 
sub2 = pd.read_csv(COMP + 'sample_submission.csv')
sub2['Irrigation_Need'] = dfs.apply(majority_vote, axis=1)
sub2.to_csv('submission_majority_vote.csv', index=False)
print(f"Majority dist: {sub2['Irrigation_Need'].value_counts().to_dict()}")
print("Saved submission_majority_vote.csv")

All 4 agree: 269,744 / 270,000
Disagree:    256

Dist: {'Low': 159625, 'Medium': 100444, 'High': 9931}
Saved submission_voting_v2.csv
Majority dist: {'Low': 159639, 'Medium': 100479, 'High': 9882}
Saved submission_majority_vote.csv
